# NYC Taxi Lakehouse

Monthly NYC yellow-taxi processing with PySpark, Delta Lake and SQL.
The pipeline retains source provenance, separates rejected records and
reconciles persisted layers before recording a successful completion.

[Project overview and results](https://hudaifa-ahamed.netlify.app/projects/nyc-taxi-lakehouse/)
| [Interactive dashboards](https://hudaifa-ahamed.netlify.app/projects/nyc-taxi-lakehouse/dashboards/)

## Data Flow

```text
TLC monthly Parquet -> Landing Volume -> Bronze
                                         |
                                  Quality checks
                                   /          \
                             Quarantine      Silver
                                              |
                                     Gold fact + dimensions
                                              |
                                   Persisted reconciliation
                                              |
                                 SQL analytics + public aggregates
```

Bronze retains source fields and ingestion metadata. Silver standardizes
accepted records. Gold adds shared date and zone dimensions. The selected
source month's data is replaced without appending duplicate rerun rows;
source-row multiplicity is retained because no unique trip key is assumed.

## Notebook Responsibilities

| Notebook | Responsibility |
| --- | --- |
| 01 | Parameters, names, date policy and shared helpers |
| 02 | Land and checksum a monthly file; replace Bronze |
| 03 | Classify quality rules and replace quarantine |
| 04 | Standardize accepted rows and replace Silver |
| 05 | Merge dimensions and replace the monthly Gold fact slice |
| 06 | Reconcile persisted counts, amounts, provenance and references |
| 07 | Source-period analytical queries and quality observations |
| 08 | Bounded planning and sequential monthly execution |
| 09 | Generated checksummed stages and controller modules |
| 10 | Current-policy coverage and operational history |
| 11 | Optional publication connectivity/secret-metadata diagnostic |
| 12 | Generated aggregate validation helpers |
| 13 | Prepare validated monthly aggregates without publishing |
| 16 | Publish aggregates after successful processing |

The runner executes stages 01-06 in one per-month Python namespace. Plan
mode creates operational metadata if necessary but performs no source probes
or business-table writes. Generated notebooks must be rebuilt from canonical
inputs rather than edited directly.

## Interpretation

- Source period identifies a monthly input file, not every pickup's date.
- The default one-day boundary grace retains nearby-month pickups as warnings.
- Bronze = Silver + quarantine; Silver = Gold. Reconciliation also checks
  per-row rounded cents, nulls, invalid values, provenance and dimension keys.
- Source fare and total amounts are measures from the input, not net revenue.
- Shared dimension merges and monthly layer writes are separate transactions.
- Public dashboards consume validated aggregates, not workspace credentials
  or raw trip records. Summary refresh is separate from daily Job health.